# BirdCLEF OpenVINO Submission Notebook

单文件自包含版提交 Notebook。默认使用以下 Kaggle 路径：
- 数据：`/kaggle/input/competitions/birdclef-2026`
- OpenVINO wheel：`/kaggle/input/datasets/rulerbug/openvino/openvino-2026.1.0-21367-cp312-cp312-manylinux_2_28_x86_64.whl`
- 权重：`/kaggle/input/datasets/rulerbug/bird-model`

`DRY_RUN` 逻辑按你给的 BirdSound 模板保留：优先读隐藏 `test_soundscapes`，拿不到时自动回退到 `DRY_RUN_AUDIO_DIR`、`train_soundscapes`、`unlabeled_soundscapes`。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("BIRDCLEF_DATA_DIR", "/kaggle/input/competitions/birdclef-2026")
os.environ.setdefault("OPENVINO_WHEEL", "/kaggle/input/datasets/rulerbug/openvino/openvino-2026.1.0-21367-cp312-cp312-manylinux_2_28_x86_64.whl")
os.environ.setdefault("WEIGHTS_ROOT", "/kaggle/input/datasets/rulerbug/bird-model")

wheel = Path(os.environ["OPENVINO_WHEEL"])
if wheel.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(wheel)], check=True)
else:
    print(f"warning: OpenVINO wheel not found: {wheel}")


In [ ]:
import base64
import zlib

SCRIPT_B64 = """
eNqdGmtz2zbyu38Fj/eFbGn61WQyulPnnMRuPZc4GcfpzZ2q4dAkJLOmQJYgbbk5//fbB0CAlJS4l3EkAdhd7C4W+wCwaKqVlySLru0akSResaqrpvVSKas2
bYtKqr093cdfZXFjOiplfjWdrB9NQz32/W2xEnsLnCJP2zQrU6WEMnP0XQyx6GTWVlXZj5dNl2Rpdqsp1Gl7C5Ob0Y/Q5IH2sS7k0vSfyseeYwBvKpWapuxW
9aOXKk/WvQi1kPeFrLC3uje9dSpz6IC/Ou+lqjqZL4pSYLda9BJWTXY7aMRSxiQLKC8tEfp8MJ52eVFpzn/PV3HatZVhHjv29vauzj5+SN5eXHlTEjOoVIxs
NpWMl6INfDPuRzwO6weMJUkYN0JV5b0IwrhOGyFbNTuah+Hev84ufvr5+lNy9eHD9S6iLgwQ9g/u0uWyFAeFrLv24KZo8qwUi+PD45P9B1Esb1vlA+W3p9en
X2P19cXV2zfvzs4TA7ib9D7SRpofPl8nH0+vf95F04y7pB6q5g6s4EB1N6tCKdB9nKl7pLb39uz89PO76+T9h7dn7z4B0dmeB/98JfLk/lj5kW3enMhBWwlQ
qBTr9vhla/ozKV00bBJQe/JD7na6tKh9SO353h4xklyevj+z3Kxj1TZFHYTUWlSNt/YK6Y1Fd1BR+siPf6sKGQxFDMNY1WUB4JHP9IqFM8FcM6AXvGdhUVZp
G/Rwz+JE03B5mflH8aE/977zSiEDh+Hwm3xB0+Jo0qH3l+kGpQlhN2mhhPdLWnbirGmqZsSUt+pU692m98Jrb4Wn0pVAQsv2Fjelq0kwkg8fzy5/ubiEbXX2
y8WbM1DKhtUNIVDkNx8/A+7r0+s3PyefLv6DWIVsN/dAD4BIJ8e0b67+nVx9vkzOL96RFWzDG8Ag6gDz9PPbC+MldmH2MIjth72qzy4/nb1//e4seX929dM2
YYcAiL0SqQQKZfUgGqCwBxjZbV40gXFGIS4fTGA7PIghaDoQEGL03rxqphUXErZXGxxGQyygTc5xVeVdKQC2EXVTZUL1gQF7wLsl2WLpeX+FWX5PJ97ZD4fH
4DpzsQD33SiRCIn7N6tkHjTVQ1LkE5wHJxMr+hl6+z+i3r3/epeVFMwd0F4Ua9DIwv+CkE+JbywVpWFKoMe0adVDAc6JEbRFklUKCKWSSFJf2zxuDOJqM6kZ
mram4U3mvC3EOhN161j2duosrZVTJbBbKRQEYOx1KSYQv+K3EGbPGzD+sehlodoZcDLXkqPWcvQGG/rD9WF2QyYSkl/gLlphmm7mc48/j1MFUVkgYjhnkYA9
oK1g/UQeCMIXiKqnBf1CU5GSUThWhJaWkIsFf4sSNj3yHjSpXIrgReS9hP8v0HBIITiWUJQlXaiAf4OtTiieWOGxpaXX69uDxmINECrYXNiZFmjdokBf/Lha
LnGDxA/pPX2v6hP6XpRpxh0/pP6TK4/WQk1aqFELdt6iFQ1uqxBZquNC8YKGkJLl0FbdAkzFbENERUaM6I1IcxY9oA1H8kZ6eZImbcEkYMVJA7KOIctpmlSb
J6EBbIPLtIiRFC08EoJlz3FBpz4FCXBhkZeWD+mjSo7z6XkKS9I7dKIDpIuVN516x1Z/NADEGQD9SZCCkqdHPSrMDc7e5XYDWad0mOgQGC+usTeQSTMIHFdN
sUxUM1Ww6WG/gl+jlqU+sDFATRWpI9CaYIkdklrJWVPVCWy0GnQt7tOS4SeOQiMdZ3YrW0c6wgy9vxv4sa0BGsxiGEJXqQPYvoMOooKvhLWBvQqOSbbgpjf0
EXlZVT+Olsph4cddLND4bMKj8+cQdhG/Ba9d2Lpt0qxNHgqZVw/b9KmdESl0054jjzEZhCbaqnWG0tG2wYQ+sIiQsLi2ERq3NQDXfGyFpaAA0Kt0jSuFqPt6
SgbIyqI29j8j6AkAzV2lbdgW4kQ9FW2A1QryZpHkomxTFahaZEtw8BNdf1wLCQ6GkBLHDmHmF2wpFAKg6UMcLYsMJPBJXS46KywX90Um0CHoOWLu4UFc2cEY
drAqbtPhkCr+EDq3NX3uMOxmxAiOIm8f/lNjtn+kw6EE0MBKA1o9Cr2DA+9Y8ygrJCZhUQLpfY+D8OsYPnTrwDsZz3xO+8p0wNaSkSfNRsIPnvlONFKUAM+q
STni7CMwko7wjzUy5S/jNegTy7FaQE5t1UCCHc0JUfu9qmthNYkn2MD3Ry5bPH/kLcH6ajXdoIOykfyuDTHBXqf0aUxnld5puwmo+EpaXu3B2u8wBheBNoVt
xrB/paorJYKTyDsOt8GPrNYdDf8v+n0uZeG0lMUqXRox18+SjWCTI9q9vYbWoTN2PBzTCANOmCzsp2C2jgxJ8+MYFh1iIka7vb1/9GcbAXgL3BxTTnpojUSZ
VDXs60Y7t1vwCc5GBuNLAEbpxiJZFVJ7PWql674lk8Wi5SjEciIeGJGxaPbRpFxIRVYqfi/KTzDeNhXaWdAHA8fbYTzt+y1rU/vTDjOjU/6y3cTylD4Hnel6
Sp8uARBgSp+2MxMSkqTpddMJ24kecxQGHTIg3NRXZSrFo9MNKldFLvIRKVwAlaWlGKFoc7hJeoXt1OPpCgvdLhfX1dvXgeLcqcakDTKnFjSV30xfHQ6sRy9N
NJhAGzSnh22VkGVvC5BQC03w7Gt3aodbcGvmEjnxDchAXbYQTZJ3DZ3/gRvFzk9QmDG7694dYpmW0KGaziI4xgdh3En1eycE+Hwt41bhcEdpW+cpSIzYMSRq
a+uh32wz5idYioYA+7DRheniL7OFda87OXnZPhMiKtCblpi0vjq0aZBGJXDw+bBo4HNfHXJ2UG6gHr94sYFLXwc4pLGUk9jyIcZl1V7gDluhaed8mrHwP0vV
1TUVC8400y/295M/EBC/HOU7mTUO9GEVs2xa6RUkyOB8Njg+ByOALVZXJWz4gFeOvFQwXhBDok9Cb4qykCJtqEIoljLJKohhjRqlnmxoCWwdqVBvJxs8uE7c
rlWv8h4ZZDnaqnITfY+iAYIbe009RtrpLTbO6g6MmC07fF4Kq4rlqiryRNYYcewG3Lof9bTBUXwIdkFf3yOQWNfB/hrPyZ41J+sXvII99MSjhHzkGfjUfcIl
L2R+874cyqrS7d/Ka0W5CTLSlWVA9LUhHc6pHAkcYliLHIn9l9uKJyRWVtVdhznwlzK9ESVEpnxNRTB8AzHso5IWVC8w1gSa+fCpP4z8bSucy4M1BixxDCxP
bce0bLNJ5P02B45QMGww3Izw5vNRXmWyKAGzJYhR0DG/IrUYVVoNgoL0UbkeImVs1zOErOyOzl9ghBpMFLYRFsmHz63pxkd7kOaDy/S3lZbQH+hp/9QsLFMC
zDO3pm7Wsu5a/AGa0zigiolDh+2OVbcKtD0dHY9LdRw0GvvOIYYriJkUf86fLRavay5ayH2SRSFTyABaDLX8G11bmfCRijk+g+9Jb5UEjGYW+IABbEqBRz8Q
bcDt3GROg3/D0g7G+jY0HQNuqzsh6TDy4AtN8XTgu9bN4zDtmE30fWWaicD/9Ve6JnGpup4Pibq6tVz2R0p0o5SsV0w54ElkuhL6QHWQenzsz3kNArA/uH46
IG9sLt4Y5ACPWy3hpxiQ+3NXQ2j3kZyBYIdlKOdF84y5uTJPZV7k4EicM0qXTrwsq5vA/474CkP3xNCijmP6eVEKiOvnmFiZiH5ZeR+ALt4jINdgOzBIFuSK
70GngC6XBRPpzUFBPy34Yawo7G0qfXpXnYSwq7P+4ZKxyHrV9MnEOEY4O1ZXEwwIRRyIxal2aQlwzcltXTQtPO0gAyXKRcR1mvpmbNSH8Nu8KZ5Nz229wNuu
oSsGLsz5fIxLSwWuw168jKz/Jm2zW1NjKj6L8Saa3PcO3rMOvMy/sloWdCiMAsdDVQU0ZTijIdbWfIBMMsdpDUueB0424fhXpr/FvYbhFu8OZRAe7UgMj8NI
onf2TVeUedKQnWzuau4YWABK3sB4dR+/gR+0fK6Z4X3M1D4TiPlXwhc5YPzA0aJYqtg1dT+Mb1JVZHiZY+LgEo+OtvhiyvpHji50ZnbuhRA2Ylrh2Bt9xaOR
Mwu3WDqdXzTCLKpeUzweN1TA3EY3hKPjnSFBbQPBsAZkbdqy23I2dZi0hfBiOUVBbQfv4+mMOAvt7S2qDu/ZKUOaWwRe36m72A61AcfTYdOpokmSKX+ZSlnf
xLEH4CsMtrSJFjLydh/z4l3WxN5P/al8lcoG5aZRPaOzwYYb1dSDMX3Bs3Eo3XMYeSxNbApXpzmsn8Nog7KFHY4NvQndkOnjZsp1e9X0YOxBnpWz4Uqg7fPU
xjOzsobH9juKCisi29hgUfr7p066eLRA9m6VD5Fwo+Z0vZRk6j7oH5AceL4+ZRq/4eBrVDsbxoGM9JPZy0fYmmW3knRPmGFR2F9GMrpQrc4I3Bmpm572qAxq
GuVbYLo7pBun0XWiIaXTAJXkzSO6UaybUeFudmApOVWJi4EnT3bDmaCO5I2l0yXlMPQB8Y0b/mGAG1IyYYVe02xgOtFjhAZbANEGlGcb5jzQJ2WPrkKjr8J3
kgotfGqzE8fKHQ6C/4BZcnJD9e1QyM51HaAP9yLa1JDAZEMsx8IGlLYDGi5GZGcTKoaOIm/w+COcb1CpGzy12+gm5fi3RZ5DYTC2br7ivk8LUHop/uaBGe6j
GVYSCnKs5HvOwiePOaSnGF+MbE9blpOfDnXqdnSKut2rUfYFm//ONeZdO8Xm0hAz8C0hp9EbBDGt3iUyn31qYbA/opt0WfXSuxAPohFWQ7Hnb871SbSbu8/D
5xBAq73lNwzipqru0Cb/Sc/TPOvSKKbH/sjx65SjQe+o0yt9XcZel3zejoRtmKtxksahfxcMMvZHUbtPqiJv+OpqziylZZmAH3WjrnVFOPjNVJ1yaoo9uIAx
fjiP3CglA27w4aNjfnhG3qbldGSUeJeisqnv+opw843A6BXE4L54qvUJVZOO3dY+nRALVL7yriYivmN6CWMzbwo3tFBQzPYAT8kXTenJ/0pMn49yPtIrPcbZ
TKEib5iMIP4wHPPzHAKm6ohlng8KLDolH59jOZNH3kxHfG0022laBRhbMWFDK2QIMChzsKFt/x7fOum0za1cehxbvZBVdTecR/SvmwKmgOZPecB0kJ5oHOfN
mUkPop7v0ElS9Ep+7V1T3Fb0CmnwmgLmCPHMjX4TMj/cUaLFse2UCIMg+rkdq2ZRkXkASSAZFevAkMH3eNnMos3x1reHclWFk9vZocZ11DM3cDO3UyO5XTG+
RggOY1DfUbztUI1nxPwQn1uDVyedK5niMbp8NJ8EZV7x6pfK8eoOnz3pZ8scSzw68UmqO2raVYTsFFNHQwHPFkBeN91FO8YXDtrH+7g+Tt5Fb8h865T9PRtS
IQu9F2gXljwpH8+82Y1GeFoVOO7M2zduDi+HXh5O4uPFk4fHrqFDmHSB6zU5ibzJqzkmzcBWQg46SeioNklWmEAl+rx2nE/v/Q90iR/h
"""

script = zlib.decompress(base64.b64decode(SCRIPT_B64.encode("ascii"))).decode("utf-8")
exec(compile(script, "kaggle_submit_notebook.py", "exec"), globals(), globals())
